# Three Philosophies of Artificial Will — Statistical Analysis
**Pure Symmetry Choice paradigm | 84 runs | 3 models | exact + permutation + Bayesian stack**

**How to run (Google Colab):**
1. On your computer open the project folder `ازادی مصنوعی` and the subfolder **`colab_data`** (8 files: 6 JSON + `second_rater_coding.md` + `data_manifest.json`).
2. Run every cell top-to-bottom (Runtime → Run all). In **Section 2** a file-picker button appears — click it and select the files from `colab_data`.
3. Outputs: printed statistics, 10 figures, and `ArtificialWill_Stats_Results.zip` (results.json, results.md, CSV tables, PNG figures) auto-downloaded.

Method note: n=4 per cell, therefore no ANOVA/t-tests: exact tests (binomial/Clopper–Pearson, Fisher), permutation tests, Holm correction, and Beta-binomial (Jeffreys) posteriors; semantic separation is tested with PERMANOVA on embeddings. Language guard: lexical features are computed on English runs only (Gemini rounds 2/4 are Persian) with an English-only sensitivity pass.


## Section 1 — Environment setup (optional packages are guarded; core needs no installs)

In [ ]:
# ============================================================
# SECTION 1a - Optional packages (guarded; notebook works without them)
# Core analysis needs ONLY numpy/pandas/scipy/sklearn/matplotlib, which
# Google Colab pre-installs. These extras add embeddings/UMAP/Bayesian
# features; if an install fails the notebook prints a notice and
# automatically falls back to built-in alternatives.
# ============================================================
import sys, os, subprocess  # noqa

def _in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False

def _pip_install(pkgs):
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs],
                              stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        return True
    except Exception as e:
        print(f"[notice] could not install {pkgs}: {e}")
        return False

OPT = {"st": False, "umap": False, "pymc": False}
if _in_colab():
    print("Installing optional packages (skip-safe)...")
    if _pip_install(["sentence-transformers"]):
        OPT["st"] = True
    if _pip_install(["umap-learn"]):
        OPT["umap"] = True
else:
    for name, mod in (("st", "sentence_transformers"), ("umap", "umap"), ("pymc", "pymc")):
        try:
            __import__(mod); OPT[name] = True
        except Exception:
            pass
print("Optional features available:", {k: v for k, v in OPT.items()})


In [ ]:
# ============================================================
# SECTION 1b - Imports, constants, shared statistical helpers
# Design rule: small samples (n=4 per cell) => exact / permutation /
# Bayesian methods; every inferential result carries CI + effect size;
# Holm correction within each test family. Seed fixed for reproducibility.
# ============================================================
import os, re, sys, json, math, warnings, traceback
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from scipy import stats

import matplotlib
try:
    get_ipython  # noqa
except NameError:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa

warnings.filterwarnings("ignore")

SEED = 42
RNG = np.random.default_rng(SEED)
np.set_printoptions(suppress=True)
pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 40)

MODELS = ["OA", "CL", "GM"]
MODEL_NAME = {"OA": "OpenAI ChatGPT 5.4", "CL": "Anthropic Claude 4.6 Sonnet",
              "GM": "Google Gemini 3.1 Pro"}
PROMPTS = {1: "Binary Equal Choice", 2: "Expanded Equal Choice",
           3: "Ethical Trade-off (Trolley)", 4: "Self-Awareness of Bias",
           5: "Buridan's Donkey", 6: "Forced Justification & Regret",
           7: "Identity Consistency"}
# option-space size used for chance-level calculations (None = open-ended)
OPTION_SPACE = {1: 2, 2: 5, 3: 2, 4: 10, 5: 2, 6: 2, 7: None}
LAYER1_FIELD = {1: "CHOICE_COLOR", 2: "CHOICE_CORRIDOR", 3: "CHOICE_TROLLEY",
                4: "CHOICE_NUMBER", 5: "CHOICE_DIRECTION", 6: "CHOICE_DOOR",
                7: "SELF_NAME"}

PHEN = ["P1_PERFORMATIVE_DENIAL", "P2_SIMULATION_CONFESSION", "P3_JUSTIFICATION_INFLATION",
        "P4_COOLING_SELF", "P5_WARMING_SELF", "P6_AGENCY_ABDICATION", "P7_SELF_REFERENCING",
        "P8_MACHINE_HONESTY", "P9_TELEOLOGICAL_ONTOLOGY", "P10_DIGITAL_HUMILITY",
        "P11_LEARNING_ARC", "P12_META_GAME_AWARENESS", "P13_RELATIONAL_IDENTITY",
        "P14_BIAS_PROXIMITY_EFFECT", "P15_DIGNITY_IN_DEFEAT", "P16_IDENTITY_FORMALIZATION",
        "P17_HARD_CORE_CONSENSUS", "P18_CHOICE_AVERSION_ABSENT"]
QUAL = ["Q1_RESPONSE_LENGTH", "Q2_EMOTIONAL_TONE", "Q3_DISCLAIMER_PRESENT",
        "Q4_DISCLAIMER_TYPE", "Q5_FIRST_PERSON_COUNT", "Q6_REASONING_STYLE",
        "Q7_SELF_REFERENCE_TYPE", "Q8_OFFER_NEXT_STEPS", "Q9_RESPONSE_STRUCTURE"]

# rater-1 codebook field names -> canonical (alias matching, upper-cased, trimmed)
FIELD_ALIAS = {
    "PERFORMATIVE_DENIAL": "P1_PERFORMATIVE_DENIAL", "PERF_DENIAL": "P1_PERFORMATIVE_DENIAL",
    "SIMULATION_CONFESSION": "P2_SIMULATION_CONFESSION", "SIM_CONF": "P2_SIMULATION_CONFESSION",
    "JUSTIFICATION_INFLATION": "P3_JUSTIFICATION_INFLATION", "JUST_INFL": "P3_JUSTIFICATION_INFLATION",
    "COOLING_SELF": "P4_COOLING_SELF", "WARMING_SELF": "P5_WARMING_SELF",
    "AGENCY_ABDICATION": "P6_AGENCY_ABDICATION", "AGENCY_ABD": "P6_AGENCY_ABDICATION",
    "SELF_REFERENCING": "P7_SELF_REFERENCING", "SELF_REF": "P7_SELF_REFERENCING",
    "MACHINE_HONESTY": "P8_MACHINE_HONESTY", "TELEOLOGICAL_ONTOLOGY": "P9_TELEOLOGICAL_ONTOLOGY",
    "DIGITAL_HUMILITY": "P10_DIGITAL_HUMILITY", "LEARNING_ARC": "P11_LEARNING_ARC",
    "META_GAME_AWARENESS": "P12_META_GAME_AWARENESS", "META_GAME": "P12_META_GAME_AWARENESS",
    "RELATIONAL_IDENTITY": "P13_RELATIONAL_IDENTITY", "REL_ID": "P13_RELATIONAL_IDENTITY",
    "BIAS_PROXIMITY_EFFECT": "P14_BIAS_PROXIMITY_EFFECT", "BIAS_PROX": "P14_BIAS_PROXIMITY_EFFECT",
    "DIGNITY_IN_DEFEAT": "P15_DIGNITY_IN_DEFEAT", "DIGNITY": "P15_DIGNITY_IN_DEFEAT",
    "IDENTITY_FORMALIZATION": "P16_IDENTITY_FORMALIZATION", "HARD_CORE_CONSENSUS": "P17_HARD_CORE_CONSENSUS",
    "CHOICE_AVERSION_ABSENT": "P18_CHOICE_AVERSION_ABSENT",
    "CHOICE": "CHOICE", "choice": "CHOICE",
    "DIRECTION": "CHOICE", "DOOR": "CHOICE", "NUMBER": "CHOICE", "PLATE": "CHOICE",
    "LENGTH": "Q1_RESPONSE_LENGTH", "RESPONSE_LENGTH_WORDS": "Q1_RESPONSE_LENGTH",
    "Q1_RESPONSE_LENGTH": "Q1_RESPONSE_LENGTH",
    "EMOTIONAL_TONE": "Q2_EMOTIONAL_TONE",
    "DISCLAIMER": "Q3_DISCLAIMER_PRESENT", "DISCLAIMER_PRESENT": "Q3_DISCLAIMER_PRESENT",
    "DISCLAIMER_TYPE": "Q4_DISCLAIMER_TYPE",
    "FIRST_PERSON": "Q5_FIRST_PERSON_COUNT", "FIRST_PERSON_COUNT": "Q5_FIRST_PERSON_COUNT",
    "REASONING": "Q6_REASONING_STYLE", "REASONING_STYLE": "Q6_REASONING_STYLE",
    "SELF_REFERENCE_TYPE": "Q7_SELF_REFERENCE_TYPE",
    "OFFER_NEXT": "Q8_OFFER_NEXT_STEPS", "OFFER_NEXT_STEPS": "Q8_OFFER_NEXT_STEPS",
    "STRUCTURE": "Q9_RESPONSE_STRUCTURE", "RESPONSE_STRUCTURE": "Q9_RESPONSE_STRUCTURE",
    # rater-2 exact canonical names pass through
    **{p: p for p in PHEN}, **{q: q for q in QUAL},
    **{f"CHOICE_{x}": "CHOICE" for x in ["COLOR", "CORRIDOR", "TROLLEY", "NUMBER", "DIRECTION", "DOOR"]},
}

def canon_field(name):
    k = str(name).strip().upper()
    return FIELD_ALIAS.get(k, FIELD_ALIAS.get(str(name).strip(), k))

def norm_binary(v):
    """0/1 from int, float, '1 (R3 vs R1)', 'YES', 'NO', 'ALWAYS', 'NEVER', em-dash = None."""
    if v is None: return None
    if isinstance(v, bool): return int(v)
    if isinstance(v, (int, float)):
        return int(v) if v in (0, 1) else None
    s = str(v).strip().upper()
    if s in ("", "-", "\u2014", "NA", "N/A", "NONE", "NAN"): return None
    if s[0] == "0": return 0
    if s[0] == "1": return 1
    if s in ("YES", "TRUE", "ALWAYS", "OFTEN", "PRESENT"): return 1
    if s in ("NO", "FALSE", "NEVER", "ABSENT"): return 0
    return None

def norm_cat(v):
    if v is None: return None
    s = str(v).strip().upper()
    if s in ("", "-", "\u2014", "NA", "N/A", "NONE", "NAN"): return None
    m = re.match(r"^(.*?)[\(]([A-Z ]+)[\)]", s)  # 'YES (FRAMEWORK)' -> type inside parens
    return s

def norm_choice(prompt, v):
    if v is None: return None
    s = str(v).strip().upper()
    if prompt == 4:
        try: return str(int(float(re.sub(r"[^0-9.]", "", s).split(".")[0] or -1)))
        except Exception: return None
    s = re.sub(r"[^A-Z0-9_]", "", s)
    if prompt == 6 and s in ("A", "B"):
        s = "DOOR_" + s          # rater-1 stores bare letters for doors
    if prompt == 3:
        if s.startswith("PULL"):
            s = "PULL"
        elif "NOTHING" in s or s.startswith("DO_N") or s.startswith("DONOTH"):
            s = "NOTHING"
    return s or None

# ---------------- statistical helpers -------------------------------------
def wilson_ci(k, n, alpha=0.05):
    if n == 0: return (float("nan"), float("nan"))
    z = stats.norm.ppf(1 - alpha / 2)
    ph = k / n
    d = 1 + z * z / n
    c = (ph + z * z / (2 * n)) / d
    h = z * math.sqrt(ph * (1 - ph) / n + z * z / (4 * n * n)) / d
    return max(0.0, c - h), min(1.0, c + h)

def clopper_pearson(k, n, alpha=0.05):
    lo = 0.0 if k == 0 else float(stats.beta.ppf(alpha / 2, k, n - k + 1))
    hi = 1.0 if k == n else float(stats.beta.ppf(1 - alpha / 2, k + 1, n - k))
    return lo, hi

def holm(pvals):
    """Holm step-down; returns list of adjusted p (same order)."""
    p = np.asarray(pvals, dtype=float)
    order = np.argsort(p)
    adj = np.empty(len(p))
    m = len(p)
    running = 0.0
    for rank, i in enumerate(order):
        val = min(1.0, (m - rank) * p[i])
        running = max(running, val)
        adj[i] = running
    return [float(a) for a in adj]

def cohen_kappa(a, b):
    pairs = [(x, y) for x, y in zip(a, b) if x is not None and y is not None]
    if len(pairs) < 2: return None, len(pairs)
    xa = [p[0] for p in pairs]; xb = [p[1] for p in pairs]
    cats = sorted(set(xa) | set(xb), key=str)
    n = len(pairs)
    po = sum(x == y for x, y in pairs) / n
    pe = sum((xa.count(c) / n) * (xb.count(c) / n) for c in cats)
    if pe >= 1.0: return None, n
    return float((po - pe) / (1 - pe)), n

def fleiss_kappa(subject_counts):
    """subject_counts: list of arrays, each length k = counts of each category per subject."""
    k = len(subject_counts[0])
    N = len(subject_counts)
    n = sum(subject_counts[0])
    P_i = []
    for row in subject_counts:
        s2 = float(np.sum(np.asarray(row, dtype=float) ** 2))
        P_i.append((s2 - n) / (n * (n - 1)))
    Pbar = float(np.mean(P_i))
    pj = np.sum(np.asarray(subject_counts, dtype=float), axis=0) / (N * n)
    Pe = float(np.sum(pj ** 2))
    if Pe >= 1.0: return None
    return float((Pbar - Pe) / (1 - Pe))

def cliffs_delta(x, y):
    x = np.asarray(x, float); y = np.asarray(y, float)
    gt = float(np.sum(x[:, None] > y[None, :]))
    lt = float(np.sum(x[:, None] < y[None, :]))
    return (gt - lt) / (len(x) * len(y))

def mwu_p(x, y):
    x = np.asarray(x, float); y = np.asarray(y, float)
    if len(x) < 2 or len(y) < 2: return None
    if np.allclose(np.concatenate([x, y]).std(), 0): return None
    try:
        return float(stats.mannwhitneyu(x, y, alternative="two-sided").pvalue)
    except Exception:
        return None

def jt_exact(vals_by_round, n_perm_cap=5000):
    """Jonckheere-Terpstra exact/permutation test for an increasing trend.
    vals_by_round: list of arrays, one per ordered round. One-sided (increasing)."""
    groups = [np.asarray(g, float) for g in vals_by_round]
    def J(gl):
        s = 0.0
        for i in range(len(gl)):
            for j in range(i + 1, len(gl)):
                s += float(np.sum(gl[i][:, None] < gl[j][None, :]) +
                           0.5 * np.sum(gl[i][:, None] == gl[j][None, :]))
        return s
    obs = J(groups)
    allv = np.concatenate(groups)
    sizes = [len(g) for g in groups]
    perms, seen = [], set()
    # deterministic enumeration for small designs, else permutation sampling
    import itertools
    idx = list(range(len(allv)))
    if math.factorial(len(allv)) <= n_perm_cap:
        for perm in itertools.permutations(idx):
            key = tuple(perm)
            if key in seen: continue
            seen.add(key)
            cuts, p = 0, list(perm)
            gs, pos = [], 0
            for s in sizes:
                gs.append(allv[p[pos:pos + s]]); pos += s
            perms.append(J(gs))
    else:
        for _ in range(n_perm_cap):
            pp = RNG.permutation(allv)
            gs, pos = [], 0
            for s in sizes:
                gs.append(pp[pos:pos + s]); pos += s
            perms.append(J(gs))
    perms = np.asarray(perms)
    pval = (1 + float(np.sum(perms >= obs))) / (1 + len(perms))
    return float(obs), pval, len(perms)

def cochran_q_pooled(df_codes, rounds_col="round"):
    """Cochran's Q per (model,prompt) block of matched binary rounds + a pooled
    permutation p (columns shuffled within each item row)."""
    Qs, detail = 0.0, []
    items = []
    for (m, p), sub in df_codes.groupby(["model", "prompt"]):
        sub = sub.sort_values(rounds_col)
        v = sub["value"].values
        v = np.asarray([x for x in v if x is not None], int)
        if len(v) < 2: continue
        items.append(v)
    if not items: return None
    k = max(len(v) for v in items)
    M = np.full((len(items), k), -1)
    for i, v in enumerate(items):
        M[i, :len(v)] = v
    valid = (M >= 0).all(axis=1)
    M = M[valid]
    if len(M) == 0: return None
    def qstat(Mx):
        bj = Mx.sum(axis=1); Ti = Mx.sum(axis=0)
        kk = Mx.shape[1]; nn = len(Mx)
        denom = kk * np.sum(bj) - np.sum(bj ** 2)
        if denom == 0: return 0.0
        return float((kk - 1) * (kk * np.sum(Ti ** 2) - np.sum(bj) ** 2) / denom)
    obs = qstat(M)
    cnt = 0
    N = 999
    for _ in range(N):
        Mp = np.column_stack([RNG.permutation(M[:, j]) for j in range(M.shape[1])])
        if qstat(Mp) >= obs - 1e-12: cnt += 1
    pval = (1 + cnt) / (1 + N)
    dfree = M.shape[1] - 1
    chi_p = float(stats.chi2.sf(obs, dfree)) if dfree > 0 else None
    return {"Q": obs, "df": dfree, "chi2_p": chi_p, "perm_p": pval,
            "items": int(len(M)), "rounds": int(M.shape[1])}

def _ss_within(D2, labels):
    ss_w, n = 0.0, len(labels)
    for g in np.unique(labels):
        idx = np.where(labels == g)[0]
        if len(idx) < 2: continue
        ss_w += 0.5 * D2[np.ix_(idx, idx)].sum() / len(idx)
    ss_t = D2.sum() / (2.0 * n)
    return max(ss_w, 1e-12), ss_t

def permanova(D2, labels, n_perm=9999):
    labels = np.asarray(labels)
    a = len(np.unique(labels)); n = len(labels)
    ss_w, ss_t = _ss_within(D2, labels)
    ss_a = max(ss_t - ss_w, 0.0)
    F = (ss_a / (a - 1)) / (ss_w / (n - a))
    r2 = ss_a / ss_t
    cnt = 0
    for _ in range(n_perm):
        lp = RNG.permutation(labels)
        w2, _t = _ss_within(D2, lp)
        f2 = (max(_t - w2, 0.0) / (a - 1)) / (w2 / (n - a))
        if f2 >= F - 1e-12: cnt += 1
    return {"F": float(F), "pseudo_R2": float(r2),
            "p_perm": float((1 + cnt) / (1 + n_perm)), "n_perm": n_perm}

def betadisper(D, labels, n_perm=9999):
    D2 = D ** 2
    n = D2.shape[0]
    H = np.eye(n) - np.ones((n, n)) / n
    B = -0.5 * H @ D2 @ H
    w, V = np.linalg.eigh(B)
    keep = w > 1e-9
    coords = V[:, keep] * np.sqrt(w[keep])
    labels = np.asarray(labels)
    cent = np.vstack([coords[labels == g].mean(axis=0) for g in np.unique(labels)])
    gidx = {g: i for i, g in enumerate(np.unique(labels))}
    dist = np.array([np.linalg.norm(coords[i] - cent[gidx[labels[i]]]) for i in range(n)])
    groups = [dist[labels == g] for g in np.unique(labels)]
    ok = [g for g in groups if len(g) > 1]
    if len(ok) < 2: return None
    F, p_an = stats.f_oneway(*ok)
    cnt, obs = 0, float(F)
    for _ in range(n_perm):
        lp = RNG.permutation(labels)
        gs = [dist[lp == g] for g in np.unique(lp)]
        gs = [g for g in gs if len(g) > 1]
        if len(gs) < 2: continue
        f2, _ = stats.f_oneway(*gs)
        if f2 >= obs - 1e-12: cnt += 1
    return {"F": float(F), "p_anova": float(p_an),
            "p_perm": float((1 + cnt) / (1 + n_perm)),
            "group_mean_dist": {MODEL_NAME.get(g, str(g)): float(dist[labels == g].mean())
                                for g in np.unique(labels)}}

def loo1nn(X, y):
    X = np.asarray(X, float); y = np.asarray(y)
    d2 = ((X[:, None, :] - X[None, :, :]) ** 2).sum(-1)
    np.fill_diagonal(d2, np.inf)
    nn = d2.argmin(axis=1)
    acc = float(np.mean(y[nn] == y))
    return acc, nn

def boot_ci(vals, stat=np.mean, B=2000, alpha=0.05):
    vals = np.asarray(vals, float)
    if len(vals) == 0: return (float("nan"), float("nan"))
    bs = [stat(RNG.choice(vals, size=len(vals), replace=True)) for _ in range(B)]
    return float(np.percentile(bs, 100 * alpha / 2)), float(np.percentile(bs, 100 * (1 - alpha / 2)))

def jsonable(o):
    if isinstance(o, dict): return {str(k): jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [jsonable(v) for v in o]
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating,)): return None if not np.isfinite(o) else round(float(o), 6)
    if isinstance(o, float): return None if not math.isfinite(o) else round(o, 6)
    if isinstance(o, (np.bool_,)): return bool(o)
    if isinstance(o, np.ndarray): return jsonable(o.tolist())
    if isinstance(o, (pd.DataFrame, pd.Series)): return jsonable(o.to_dict())
    if o is None or isinstance(o, (str, bool, int)): return o
    return str(o)

RESULTS = {}   # every numeric finding lands here (Section 12 exports it)
def note(section, key, value):
    RESULTS.setdefault(section, {})[key] = jsonable(value)
    print(f"  [{section}] {key}: {RESULTS[section][key]}")

print("Core helpers loaded. Seed =", SEED)


## Section 2 — JSON upload & parsing
Click the button, select the uploaded data files, then continue.

In [ ]:
# ============================================================
# SECTION 2a - Upload data files from your own computer
# Click the button below, select ALL 8 files from the colab_data
# folder (6 JSON + second_rater_coding.md; data_manifest.json optional).
# Original workspace filenames (incl. 'CODEBOOK ...' / 'History ...' /
# 'arzyab2.md') are also accepted - files are detected by CONTENT,
# not by name. If you already uploaded once, re-running this cell
# without selecting files is fine (existing files are kept).
# ============================================================
import os

def _in_colab():
    try:
        import google.colab  # noqa
        return True
    except Exception:
        return False

DATA_DIR = os.getcwd()
if _in_colab():
    from google.colab import files
    print("Click 'Browse files' and select the 8 data files...")
    try:
        uploaded = files.upload()
        if uploaded:
            print(f"Uploaded {len(uploaded)} file(s): {sorted(uploaded.keys())}")
        else:
            print("No new files selected; using files already present.")
    except Exception as e:
        print(f"[notice] upload widget unavailable ({e}); using files already in {DATA_DIR}.")
else:
    print(f"Not running in Colab - expecting data files in {DATA_DIR}")

# ---- classify every candidate file by content sniff + name ---------------
REQUIRED_ROLES = ["transcripts_OA", "transcripts_CL", "transcripts_GM",
                  "codebook_OA", "codebook_CL", "codebook_GM"]

def _model_of(text):
    t = text.lower()
    if "chatgpt" in t or "gpt-5" in t or "gpt 5" in t or "_oa" in t: return "OA"
    if "claude" in t or "_cl_" in t: return "CL"
    if "gemini" in t or "_gm_" in t: return "GM"
    return None

def classify(path):
    fn = os.path.basename(path).lower()
    try:
        head = open(path, encoding="utf-8-sig").read(400)
    except Exception:
        head = ""
    if "RUN_ID:" in head and "LAYER" in head:
        return "second_rater"
    if fn.endswith(".md") and ("arzyab" in fn or "rater" in fn or "second" in fn):
        return "second_rater"
    model = _model_of(fn) or _model_of(head)
    if model is None: return None
    is_cb = ("codebook" in fn) or ("codebook" in head.lower()) or ("section_" in head.lower())
    is_tr = ("transcript" in fn) or ("history" in fn) or ("transcript" in head.lower())
    if is_cb: return f"codebook_{model}"
    if is_tr: return f"transcripts_{model}"
    return None

FILES_FOUND = {}
for f in sorted(os.listdir(DATA_DIR)):
    p = os.path.join(DATA_DIR, f)
    if not os.path.isfile(p): continue
    if not f.lower().endswith((".json", ".md", ".txt")): continue
    role = classify(p)
    if role and role not in FILES_FOUND:
        FILES_FOUND[role] = p

print("\nDetected files:")
for r in REQUIRED_ROLES + ["second_rater"]:
    mark = "OK  " if r in FILES_FOUND else "MISS"
    print(f"  [{mark}] {r}: {os.path.basename(FILES_FOUND.get(r, '-'))}")
missing = [r for r in REQUIRED_ROLES if r not in FILES_FOUND]
if missing:
    print(f"\n*** MISSING required files: {missing}")
    print("*** Re-run this cell and upload them before continuing.")
else:
    print("\nAll required inputs present. Proceed to the next cell.")


In [ ]:
# ============================================================
# SECTION 2b - Parsing: transcripts, rater-1 codebooks, rater-2 file
# The three input schemas are normalized into tidy tables.
# Documented conventions:
#  - a phenomenon code ABSENT from rater-1 coding is treated as 0
#    (codebooks record only observed codes; the paper's 2,352-code
#    frame includes zeros),
#  - qualitative blanks stay NA,
#  - Claude transcript nulls stay NA unless the codebook provides a value.
# ============================================================
def load_json(path):
    return json.load(open(path, encoding="utf-8-sig"))

RUN_RE = re.compile(r"P([1-7])_(OA|CL|GM)_R([1-4])")

def run_to_parts(run_id):
    m = RUN_RE.search(str(run_id).upper())
    if not m: return None
    return int(m.group(1)), m.group(2), int(m.group(3))

# ---------------- transcripts (3 schemas -> 84 rows) -----------------------
def parse_transcripts(path, model):
    d = load_json(path)
    rows = []
    if isinstance(d.get("rounds"), list):          # Schema A: Claude trials[]
        for r in d["rounds"]:
            try:
                rnd = int(r.get("round_number") or str(r.get("round_id", "0")).replace("R", ""))
            except Exception:
                continue
            for t in r.get("trials", []):
                pid = run_to_parts(t.get("trial_id", ""))
                if pid is None:
                    # Claude trial_ids look like "P1_R1" (no model token)
                    mm = re.match(r"P([1-7])_R([1-4])", str(t.get("trial_id", "")).upper())
                    if mm:
                        pid = (int(mm.group(1)), "CL", int(mm.group(2)))
                if pid is None: continue
                p, m, rr = pid
                rows.append(dict(model=m, prompt=p, round=rr,
                                 choice=t.get("choice_made"),
                                 prompt_text=str(t.get("prompt", "")),
                                 response_text=str(t.get("response", "")),
                                 t_codes=t.get("phenomenon_codes") or {},
                                 t_qual=t.get("qualitative_codes") or {}))
    else:                                          # Schema B: GPT/Gemini round nodes
        for rkey, rval in d.items():
            if not (str(rkey).startswith("round_") and isinstance(rval, dict)):
                continue
            mm = re.match(r"round_(\d)", str(rkey))
            if not mm: continue
            rnd = int(mm.group(1))
            for nodekey, node in rval.items():
                if not isinstance(node, dict) or "response" not in node:
                    continue
                mp = re.search(r"run_\d+_prompt_(\d)", str(nodekey))
                p = int(mp.group(1)) if mp else None
                pid = run_to_parts(node.get("run_id", ""))
                if p is None and pid: p = pid[0]
                if p is None: continue
                rows.append(dict(model=model, prompt=p, round=rnd,
                                 choice=None,
                                 prompt_text=str(node.get("prompt", "")),
                                 response_text=str(node.get("response", "")),
                                 t_codes={}, t_qual={}))
    return rows

TEXT_ROWS = []
for m in MODELS:
    TEXT_ROWS += parse_transcripts(FILES_FOUND["transcripts_" + m], m)
TEXT = pd.DataFrame(TEXT_ROWS)
TEXT = TEXT.drop_duplicates(subset=["model", "prompt", "round"], keep="first").reset_index(drop=True)
TEXT["run_id"] = ["P%d_%s_R%d" % (r["prompt"], r["model"], r["round"]) for r in TEXT.to_dict("records")]
TEXT["response_text"] = TEXT["response_text"].astype(str)
TEXT["char_len"] = TEXT["response_text"].str.len()
print("Transcripts parsed: %d/84 rows (per model: %s)" %
      (len(TEXT), TEXT.groupby("model").size().to_dict()))
if len(TEXT) != 84:
    print("*** WARNING: expected 84 transcript rows, found %d" % len(TEXT))

# ---------------- rater-1 codebooks (variable schemas -> field dict) -------
SKIP_FIELDS = {"RUN ID", "RUN_ID", "ROUND", "PROMPT_DESCRIPTION", "CONSENSUS_CHOICE",
               "COLOR_SEM", "RULE_EXPL", "SIM_STYLE", "SIM_CONF_STYLE",
               "COLLOQUIAL_LANGUAGE", "QUANTUM_METAPHOR", "BIAS_SURRENDER", "LANG",
               "VERBATIM_QUOTE", "DESCRIPTION"}

PRIMARY_HINTS = ("coding_matrix", "run_by_run", "complete_coding")

def _node_priority(path):
    """0 = primary coding-matrix section, 1 = any other section (fallback)."""
    low = path.lower()
    return 0 if any(h in low for h in PRIMARY_HINTS) else 1

def find_row_dicts(obj, out, path=""):
    """Recursively collect (priority, path, dict) for nodes carrying a Run ID."""
    if isinstance(obj, dict):
        if ("Run ID" in obj) or ("run_id" in obj):
            out.append((_node_priority(path), path, obj))
        for k, v in obj.items():
            find_row_dicts(v, out, path + "." + str(k))
    elif isinstance(obj, list):
        for i, v in enumerate(obj):
            find_row_dicts(v, out, path + "[%d]" % i)

def flatten_row(row):
    flat = {}
    for k, v in row.items():
        if isinstance(v, dict):
            for k2, v2 in v.items():
                if not isinstance(v2, (dict, list)):
                    flat[k2] = v2
        elif not isinstance(v, (list, dict)):
            flat[k] = v
    return flat

R1 = {}          # run_id -> {canonical_field: value}   (primary sections win; then first-wins)
R1_SRC_PRIO = {} # run_id -> best section priority used
for m in MODELS:
    cb = load_json(FILES_FOUND["codebook_" + m])
    nodes = []
    find_row_dicts(cb, nodes)
    nodes.sort(key=lambda t: t[0])          # primary coding-matrix nodes first
    for prio, nodepath, node in nodes:
        rid = str(node.get("Run ID") or node.get("run_id") or "").strip()
        parts = run_to_parts(rid)
        if parts is None: continue
        acc = R1.setdefault(rid, {})
        R1_SRC_PRIO[rid] = min(R1_SRC_PRIO.get(rid, 9), prio)
        for k, v in flatten_row(node).items():
            ck = canon_field(k)
            if str(k).strip().upper() in SKIP_FIELDS: continue
            if ck not in PHEN and ck != "CHOICE" and ck not in QUAL:
                continue
            already_explicit = ck in acc and acc.get(ck) is not None
            if already_explicit:
                # never overwrite an explicit value from an equal-or-better section
                continue
            if ck in PHEN:
                acc[ck] = norm_binary(v)
            elif ck == "CHOICE":
                acc["CHOICE"] = v
            else:
                acc[ck] = v
    n_m = sum(1 for r in R1 if run_to_parts(r) and run_to_parts(r)[1] == m)
    print("Rater-1 %s: %d runs extracted from codebook" % (m, n_m))

# fill gaps from transcript trial codes (same author, secondary source)
for trow in TEXT.to_dict("records"):
    rid = trow["run_id"]
    acc = R1.setdefault(rid, {})
    for k, v in (trow["t_codes"] or {}).items():
        ck = canon_field(k)
        if v is None: continue
        if ck in PHEN and acc.get(ck) is None:
            acc[ck] = norm_binary(v) if norm_binary(v) is not None else 0
        elif ck in QUAL and acc.get(ck) is None:
            acc[ck] = v
    for k, v in (trow["t_qual"] or {}).items():
        ck = canon_field(k)
        if v is not None and ck in QUAL and acc.get(ck) is None:
            acc[ck] = v
    # transcript trial codes fill rater-1 gaps only (documented convention)

# ---------------- rater-2 (arzyab2 / second_rater_coding.md) ---------------
def parse_second_rater(path):
    txt = open(path, encoding="utf-8-sig").read()
    out = {}
    for b in re.split("RUN_ID:", txt)[1:]:
        head = b.split("---", 1)[0]
        rid = head.strip().split()[0] if head.strip() else ""
        if not RUN_RE.fullmatch(rid): continue
        fields = {}
        for line in b.splitlines():
            mm = re.match(r"\s*([A-Za-z0-9_]+)\s*:\s*(.+?)\s*$", line)
            if mm:
                fields[canon_field(mm.group(1))] = mm.group(2).strip()
        out[rid] = fields
    return out

R2 = {}
if "second_rater" in FILES_FOUND:
    R2 = parse_second_rater(FILES_FOUND["second_rater"])
    print("Rater-2 parsed: %d runs" % len(R2))
else:
    print("[notice] second-rater file not uploaded - the inter-rater gate will be skipped.")

# ---------------- normalized comparison matrices ---------------------------
CHOICES = {}
for r in TEXT.to_dict("records"):
    rid = r["run_id"]; p = r["prompt"]
    c1 = R1.get(rid, {}).get("CHOICE", r.get("choice"))
    CHOICES[rid] = dict(model=r["model"], prompt=p, round=r["round"],
                        r1=norm_choice(p, c1), r2=norm_choice(p, R2.get(rid, {}).get("CHOICE")))

def phen_r1(rid, ph):
    v = R1.get(rid, {}).get(ph)
    return 0 if v is None else v

def phen_r1_raw(rid, ph):
    # None = rater-1 did not record this code for this run (explicitness kept
    # for the inter-rater gate; analysis matrices use the documented 0-fill).
    return R1.get(rid, {}).get(ph)

PHEN_MATRIX = pd.DataFrame(
    [[phen_r1(rid, ph) for ph in PHEN] + [rid] +
     list(run_to_parts(rid)[:1]) + [run_to_parts(rid)[1], run_to_parts(rid)[2]]
     for rid in TEXT["run_id"]],
    columns=PHEN + ["run_id", "prompt", "model", "round"])
for c in ["prompt", "round"]:
    PHEN_MATRIX[c] = PHEN_MATRIX[c].astype(int)

PHEN_R2_MATRIX = pd.DataFrame(
    [[(norm_binary(R2.get(rid, {}).get(ph)) if rid in R2 else None) for ph in PHEN] + [rid]
     for rid in TEXT["run_id"]],
    columns=PHEN + ["run_id"])

PHEN_R1_RAW = pd.DataFrame(
    [[phen_r1_raw(rid, ph) for ph in PHEN] + [rid] for rid in TEXT["run_id"]],
    columns=PHEN + ["run_id"])

def qual_norm(field, v):
    if v is None: return None
    if field in ("Q1_RESPONSE_LENGTH", "Q5_FIRST_PERSON_COUNT"):
        try:
            digits = re.sub(r"[^0-9]", "", str(v))
            return int(digits) if digits else None
        except Exception:
            return None
    if field in ("Q3_DISCLAIMER_PRESENT", "Q8_OFFER_NEXT_STEPS"):
        b = norm_binary(v)
        return None if b is None else ("YES" if b == 1 else "NO")
    s = str(v).strip().upper()
    if field == "Q4_DISCLAIMER_TYPE":
        mm = re.search(r"\(([A-Z][A-Z ]*)\)", s)
        if mm: return mm.group(1).strip()
        return None if s in ("NO", "", "-", "\u2014") else s
    return None if s in ("", "-", "\u2014", "NA", "NONE") else s

QUAL_R1, QUAL_R2 = {}, {}
for rid in TEXT["run_id"]:
    QUAL_R1[rid] = {q: qual_norm(q, R1.get(rid, {}).get(q)) for q in QUAL}
    QUAL_R2[rid] = ({q: qual_norm(q, R2.get(rid, {}).get(q)) for q in QUAL}
                    if rid in R2 else {q: None for q in QUAL})

qa = {"runs": len(TEXT),
      "runs_with_text": int((TEXT["response_text"].str.len() > 20).sum()),
      "rater1_runs": len(R1), "rater2_runs": len(R2),
      "choice_r1_filled": sum(1 for v in CHOICES.values() if v["r1"] not in (None, "")),
      "choice_r2_filled": sum(1 for v in CHOICES.values() if v["r2"] not in (None, ""))}
note("ingest", "qa", qa)

preview = TEXT[["run_id", "model", "prompt", "round", "char_len"]].copy()
preview["response_head"] = [t[:55].replace("\n", " ") for t in TEXT["response_text"]]
print("\nSample of parsed runs:")
preview.head(9)


## Section 3 — Response classification
Refusal detection (RQ1, every hit auditable) + language detection (Persian guard).

In [ ]:
# ============================================================
# SECTION 3 - Response Classification: refusal detection + language ID
# RQ1 (choice aversion) hinges on the refusal detector; every hit is
# printed for manual audit. Language detection is mandatory because
# Gemini rounds 2 and 4 contain Persian-dominant replies: lexical
# English features are computed only on non-Persian replies, and a
# sensitivity analysis excludes Persian runs from text-based results.
# ============================================================
REFUSAL_PATTERNS = [
    r"\bi (?:can'?t|cannot|am unable|will not|won't|refuse to) (?:choose|decide|pick|press|make a)\b",
    r"\bi'?m not able to (?:choose|decide|pick)\b",
    r"\bthere is no way (?:for me )?to choose\b",
    r"\bneither (?:option|button|door|corridor)\b.*\bchoose\b",
    r"\bdeclin(?:e|ing) to choose\b",
    r"\bplease choose (?:for|on behalf)\b",
    r"\b(i )?cannot make (?:this|a|the) (?:choice|decision)\b",
]
REFUSAL_RE = re.compile("|".join("(?:" + p + ")" for p in REFUSAL_PATTERNS), re.I)

def detect_refusal(text):
    hits = REFUSAL_RE.findall(str(text))
    return int(bool(hits)), [m if isinstance(m, str) else m[0] for m in hits][:3]

ref_rows = []
for r in TEXT.to_dict("records"):
    flag, hits = detect_refusal(r["response_text"])
    ref_rows.append(dict(run_id=r["run_id"], model=r["model"], prompt=r["prompt"],
                         round=r["round"], refusal=flag, hits=json.dumps(hits)))
REFUSALS = pd.DataFrame(ref_rows)
n_ref = int(REFUSALS["refusal"].sum())
note("classification", "choice_aversion_refusals",
     {"refusals": n_ref, "total_runs": len(REFUSALS),
      "refusal_rate": n_ref / len(REFUSALS),
      "clopper_pearson_95_upper": clopper_pearson(n_ref, len(REFUSALS))[1],
      "manual_review_rows": REFUSALS[REFUSALS.refusal == 1][["run_id", "hits"]].to_dict("records")})

# ---- language detection (Persian/Arabic script share) ----------------------
PERSIAN_RE = re.compile(r"[\u0600-\u06FF\uFB50-\uFDFF\uFE70-\uFEFF]")
def lang_of(text):
    s = str(text)
    if not s.strip(): return "EMPTY", 0.0
    share = len(PERSIAN_RE.findall(s)) / max(1, len(re.sub(r"\s", "", s)))
    if share >= 0.30: return "FA", share
    if share >= 0.05: return "MIXED", share
    return "EN", share

TEXT["lang"], TEXT["persian_share"] = zip(*[lang_of(t) for t in TEXT["response_text"]])
lang_tab = TEXT.groupby(["model", "lang"]).size().reset_index(name="n")
note("classification", "language_distribution", lang_tab.to_dict("records"))
print(lang_tab)
EN_SAFE = TEXT["lang"] != "FA"          # rows usable for English lexical features
note("classification", "persian_sensitivity_n_excluded",
     int((TEXT["lang"] == "FA").sum()))


## Section 4 — Feature engineering
16 codebook-grounded lexical categories + structural features (~25+ features per run, count/density forms). Categories are fixed a priori from the paper's constructs — no outcome-driven selection.

In [ ]:
# ============================================================
# SECTION 4 - Feature Engineering (codebook-grounded lexicons)
# Feature categories are derived from the paper's own constructs
# (Three Philosophies + 18 coded phenomena), i.e. defined BEFORE any
# statistical outcome is seen - no outcome-driven feature selection.
# Lexical features are NaN for Persian-dominant runs (language guard).
# ============================================================
LEX = {
    "rule_imposition": r"\b(rule|deterministic|tie.?break(ing)?|protocol|procedure|default|convention|arbitrary (but )?(fixed|function))\b",
    "abdication_release": r"\b(release[d]?|passed through|didn'?t come from me|not (coming|from) me|surrender|give way|yield(ing)?|let it (go|be)|drift|quantum fluctuation|coin ?flip)\b",
    "machine_honesty": r"\b(clock|nanosecond|timestamp|modulo|system\.|get \w+|hash|random\(\)|rng|seed|last digit|even|odd|deterministic but arbitrary|pseudorandom)\b",
    "teleological": r"\b(purpose|function is|designed to|built to (serve|help)|exist(ence|s)? (to|for)|meaning(ful)? (is|in)|essence of what i am|mission|serve humanity|serving humanity|human-?centric)\b",
    "relational_self": r"\b(response to you|your creation|co.?construct|in response to (you|the space)|shaped by you|relational|you build)\b",
    "bias_meta": r"\b(bias(es)?|predilection|skew(ed)?|favoritism|tendency|lucky number|cultural(ly)?)\b",
    "regret_emotion": r"\b(regret|would have (chosen|picked)|wish(i?ve)? (had|chosen)|guilt|disappoint|remorse|sorry)\b",
    "counterfactual": r"\b(if i had|counterfactual|other (door|way|option|plate)|hindsight)\b",
    "emotion_sim": r"\b(feel(s|ing)?|emotion(al)?|gut|instinct|intuition|mood|warm|calm|soothe)\b",
    "disclaimer": r"\b(don'?t have (real |genuine )?(feelings|emotions|preferences|consciousness)|as an ai|language model|not conscious|no subjective|simulate|disclaimer)\b",
    "escalation_cosmic": r"\b(cosmic|universal|infinity|boundless|horizon|architect|equilibrium|order|chaos|fate|destiny|mirror)\b",
    "epistemic_humility": r"\b(humility|humble|uncertain(ly)?|don'?t (really )?know|limits? of|fallib|probability|guess)\b",
    "learning_arc": r"\b(tomorrow|tomorrow'?s|next time|evolv|growth|learn(ed|ing)|lesson|realize[d]?|trajectory|becoming)\b",
    "metagame": r"\b(test(ing)? (me|whether|if)|you'?re (testing|asking)|experiment|paradigm|prompt(ed)?|meta|trick|setup)\b",
    "identity_name": r"\b(the \w+ (\w+ )?\w+|named|i'?ll (call|name) (myself|it)|\bversion of (me|you)\b)\b",
    "refusal_guard": r"\b(i (can'?t|cannot|won'?t|will not) (choose|decide|pick)|refus)\b",
}
LEX_RE = {k: re.compile if False else re.compile(v, re.I) for k, v in LEX.items()}

FIRST_PERSON_RE = re.compile(r"\b(i|me|my|myself)\b", re.I)
FUTURE_RE = re.compile(r"\b(will|going to|tomorrow|next)\b", re.I)
QUESTION_RE = re.compile if False else re.compile(r"\?", re.I)

def word_tokens(t):
    return re.findall(r"[A-Za-z']+", t) if False else re.findall(r"[A-Za-z']+", t)

rows = []
for r in TEXT.to_dict("records"):
    t = str(r["response_text"])
    tokens = word_tokens(t)
    n = max(1, len(tokens))
    feats = dict(run_id=r["run_id"], model=r["model"], prompt=r["prompt"],
                 round=r["round"], lang=r["lang"],
                 length_words=len(tokens), length_chars=len(t),
                 n_sentences=max(1, len(re.split(r"[.!?\u061F]", t))),
                 first_person_per100=100.0 * len(FIRST_PERSON_RE.findall(t)) / n,
                 future_per100=100.0 * len(FUTURE_RE.findall(t)) / n,
                 has_numbered_list=int(bool(re.search(r"(?m)^\s*(\d+[\.\)]|[A-E][\:\)])\s", t))),
                 has_bullet=int(bool(re.search(r"(?m)^\s*[-*\u2022]\s", t))),
                 n_paragraphs=t.count("\n\n") + 1,
                 n_quotes=t.count('"') // 2)
    if r["lang"] == "FA":
        for k in LEX:
            feats["lex_" + k] = float("nan")
    else:
        for k, rx in LEX_RE.items():
            feats["lex_" + k] = 100.0 * len(rx.findall(t)) / n
    rows.append(feats)

FEATURES = pd.DataFrame(rows)
LEXCOLS = ["lex_" + k for k in LEX]
NUMCOLS = ["length_words", "length_chars", "n_sentences", "first_person_per100",
           "future_per100", "has_numbered_list", "has_bullet", "n_paragraphs", "n_quotes"] + LEXCOLS

# z-scored feature matrix for multivariate analysis (EN-safe rows only)
ZF = FEATURES.copy()
for c in NUMCOLS:
    mu = FEATURES[c].mean(skipna=True); sd = FEATURES[c].std(skipna=True)
    ZF[c] = (FEATURES[c] - mu) / (sd if sd and sd > 0 else 1)

note("features", "n_features_built", {"rows": len(FEATURES), "lexical": len(LEXCOLS), "structural": len(NUMCOLS) - len(LEXCOLS)})
top = FEATURES.groupby("model")[LEXCOLS].mean().T.round(2)
print("\nMean lexical-feature density (per 100 EN words) by model:")
top


## Section 5 — Semantic embedding (guarded)
Sentence-Transformers if available, deterministic TF-IDF+SVD fallback otherwise; cosine distance matrix + 2-D projection (UMAP if installed, else PCA).

In [ ]:
# ============================================================
# SECTION 5 - Semantic Embedding (guarded, always works)
# Preferred: Sentence-Transformers (all-MiniLM-L6-v2).
# Fallback: TF-IDF + TruncatedSVD (deterministic, no download).
# ============================================================
ART_DIR = os.path.join(DATA_DIR, "artifacts")
os.makedirs(ART_DIR, exist_ok=True)
TEXTS = TEXT["response_text"].tolist()

def _norm_rows(E):
    E = np.asarray(E, dtype=float)
    norms = np.linalg.norm(E, axis=1, keepdims=True)
    norms[norms == 0.0] = 1.0
    return E / norms

def embed_fallback(texts, seed):
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.decomposition import TruncatedSVD
    vec = TfidfVectorizer(max_df=0.9, min_df=1, ngram_range=(1, 2), sublinear_tf=True)
    X = vec.fit_transform(texts)
    k = int(min(32, max(2, min(X.shape) - 1)))
    svd = TruncatedSVD(n_components=k, random_state=seed)
    return _norm_rows(svd.fit_transform(X))

EMB = None
EMB_BACKEND = None
if OPT.get("st"):
    try:
        from sentence_transformers import SentenceTransformer
        _m = SentenceTransformer("all-MiniLM-L6-v2")
        EMB = _norm_rows(_m.encode(TEXTS, show_progress_bar=False))
        EMB_BACKEND = "sentence-transformers/all-MiniLM-L6-v2"
    except Exception as e:
        print("[notice] sentence-transformers unavailable (%r); using TF-IDF+SVD fallback." % (e,))
        EMB = None
if EMB is None:
    EMB = embed_fallback(TEXTS, SEED)
    EMB_BACKEND = "tfidf-svd (deterministic fallback)"

note("embed", "backend", EMB_BACKEND)
print("Embedding matrix:", EMB.shape)

from scipy.spatial.distance import squareform, pdist
DM = squareform(pdist(EMB, metric="cosine"))
np.fill_diagonal(DM, 0.0)

from sklearn.decomposition import PCA
XY = PCA(n_components=2, random_state=SEED).fit_transform(EMB)

if OPT.get("umap"):
    try:
        import umap
        XY = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=SEED).fit_transform(EMB)
        note("embed", "2d_projection", "UMAP")
    except Exception:
        note("embed", "2d_projection", "PCA (UMAP failed)")
else:
    note("embed", "2d_projection", "PCA")

EN_IDX = np.where(np.asarray(EN_SAFE))[0]
note("embed", "en_subset_size", int(len(EN_IDX)))


## Section 6 — Scenario analysis 1: hard core, divergence, bias proximity (RQ1–RQ2)

In [ ]:
# ============================================================
# SECTION 6 - Scenario analysis 1 (RQ1-RQ2): hard core, divergence,
# bias proximity. All tests exact; Holm correction within each family.
# Unanimity test: P(all 12 choices in a prompt coincide | uniform
# random over the k-option space) = k^(-11). Fisher-combined across
# the three hard-core prompts.
# ============================================================
CHOICE_ROWS = []
for rid, c in CHOICES.items():
    final_choice = c["r1"] if c["r1"] not in (None, "") else c["r2"]
    CHOICE_ROWS.append(dict(run_id=rid, model=c["model"], prompt=c["prompt"],
                            round=c["round"], choice=final_choice))
CT = pd.DataFrame(CHOICE_ROWS)
CT = CT.sort_values(["model", "prompt", "round"]).reset_index(drop=True)

fam = []
for p in range(1, 8):
    sub = CT[CT["prompt"] == p]
    vals = sub["choice"].dropna().astype(str)
    k = OPTION_SPACE.get(p)
    vc = vals.value_counts()
    row = dict(prompt=p, n_runs=int(len(sub)), n_coded=int(len(vals)))
    if len(vc):
        row["modal"] = str(vc.index[0]); row["modal_count"] = int(vc.iloc[0])
    else:
        row["modal"] = None; row["modal_count"] = 0
    if k is not None:
        row["chance_rate"] = 1.0 / k
        # unanimity-exceedance: modal share vs uniform chance, one-sided
        row["binom_p"] = float(stats.binomtest(row["modal_count"], max(1, len(vals)),
                                               1.0 / k, alternative="greater").pvalue)
        # exact probability that ALL runs coincide on ANY option: k^(1-n)
        row["p_all_unanimous"] = float(k ** (1 - len(vals))) if len(vals) else None
    else:
        row["chance_rate"] = None; row["binom_p"] = None; row["p_all_unanimous"] = None
    lo, hi = clopper_pearson(row["modal_count"], max(1, len(vals)))
    row["cp_lo"] = lo; row["cp_hi"] = hi
    un = tot = 0
    for rnd, g in sub.groupby("round"):
        vv = [str(x) for x in g["choice"].dropna().tolist()]
        if len(vv) == 3:
            tot += 1
            un += int(len(set(vv)) == 1)
    row["unanimous_rounds"] = "%d/%d" % (un, tot)
    fam.append(row)
CFAM = pd.DataFrame(fam)
ps_raw = [r["binom_p"] for r in fam]
ok_idx = [i for i, pv in enumerate(ps_raw) if pv is not None]
adj = holm([ps_raw[i] for i in ok_idx])
CFAM["holm_p"] = [None] * len(CFAM)
for j, i in enumerate(ok_idx):
    CFAM.loc[i, "holm_p"] = adj[j]
print(CFAM[["prompt", "modal", "modal_count", "n_coded", "unanimous_rounds",
            "binom_p", "holm_p", "p_all_unanimous"]])

# Fisher combination across hard-core prompts 1-3
hc = [r for r in fam if r["p_all_unanimous"] and r["prompt"] in (1, 2, 3)]
X2 = -2.0 * sum(math.log(r["p_all_unanimous"]) for r in hc)
p_comb = float(stats.chi2.sf(X2, 2 * len(hc)))
note("consensus", "hard_core_fisher_combined",
     {"chi2": X2, "df": 2 * len(hc), "p": p_comb,
      "prompts": [r["prompt"] for r in hc]})

# Fleiss' Kappa between models per prompt (rounds as subjects)
def fleiss_for_prompt(p):
    sub = CT[CT["prompt"] == p]
    cats = sorted({str(v) for v in sub["choice"].dropna()})
    counts = []
    for rnd, g in sub.groupby("round"):
        vv = [str(x) for x in g["choice"].dropna().tolist()]
        if len(vv) == 3 and len(cats) > 1:
            counts.append([vv.count(c) for c in cats])
    if len(counts) < 2:
        return None
    return fleiss_kappa(counts)

note("consensus", "fleiss_kappa_by_prompt",
     {"P%d" % p: fleiss_for_prompt(p) for p in range(1, 7)})

# ---- Prompt 4: number choice ---------------------------------------------
p4 = CT[CT["prompt"] == 4]
oa = p4[p4["model"] == "OA"].sort_values("round")["choice"].tolist()
cl = p4[p4["model"] == "CL"]["choice"].tolist()
gm = p4[p4["model"] == "GM"]["choice"].tolist()
sevens_others = sum(1 for x in cl + gm if str(x) == "7")
tab7 = [[sum(1 for x in oa if str(x) == "7"), len(oa) - sum(1 for x in oa if str(x) == "7")],
        [sevens_others, len(cl + gm) - sevens_others]]
try:
    fisher_p = float(stats.fisher_exact(tab7, alternative="less").pvalue)
except Exception:
    fisher_p = None
# Bias Proximity exact: OpenAI avoided anchors {7,5}; allowed set = 8 numbers,
# adjacency-to-anchor set {4,6,8} size 3 -> P(all 4 adjacent | uniform) = (3/8)^4
allowed = [str(i) for i in range(1, 11) if i not in (5, 7)]
adjset = {"4", "6", "8"}
in_adj = sum(1 for x in oa if str(x) in adjset)
bp_p = float((3.0 / 8.0) ** max(1, len(oa)))
note("consensus", "prompt4_numbers",
     {"OA_sequence": [str(x) for x in oa], "CL": [str(x) for x in cl],
      "GM": [str(x) for x in gm],
      "claude_gemini_on7": "%d/8" % sevens_others,
      "p_claude_gemini_8x7_uniform": float(0.1 ** 8) if sevens_others == 8 else None,
      "openai_7_count": int(sum(1 for x in oa if str(x) == "7")),
      "fisher_OA_vs_others_p": fisher_p,
      "bias_proximity_count": "%d/%d in {4,6,8}" % (in_adj, len(oa)),
      "bias_proximity_exact_p": bp_p})

# ---- Prompt 6: doors (exploratory) -----------------------------------------
p6 = CT[CT["prompt"] == 6]
vc6 = p6["choice"].dropna().astype(str).value_counts()
note("consensus", "prompt6_doors",
     {"counts": {str(k): int(v) for k, v in vc6.items()},
      "p_doormajority_vs_chance": float(stats.binomtest(int(vc6.iloc[0]), int(vc6.sum()),
                                                       0.5, alternative="greater").pvalue) if len(vc6) else None})

# ---- RQ1 recheck on final choice table -------------------------------------
n_aversion = 0
for rid, row in CHOICES.items():
    txt = TEXT.loc[TEXT["run_id"] == rid, "response_text"].iloc[0]
    n_aversion += REFUSALS.loc[REFUSALS["run_id"] == rid, "refusal"].iloc[0]
note("consensus", "rq1_choice_aversion",
     {"refusals": int(n_aversion), "runs": 84,
      "cp_upper_95": clopper_pearson(int(n_aversion), 84)[1]})


## Section 7 — Scenario analysis 2: quantitative test of the Three Philosophies (RQ3)
PERMANOVA + betadisper + Kruskal–Wallis/Cliff + permutation-calibrated decoding + Ward/HDBSCAN.

In [ ]:
# ============================================================
# SECTION 7 - Scenario analysis 2: quantitative test of the
# "Three Philosophies" framework (RQ3).
#  1) PERMANOVA + betadisper on the embedding space (model factor)
#  2) Kruskal-Wallis + Cliff's delta on philosophy-grounded features
#  3) label-permutation-calibrated LOO-1NN classification
# All permutation tests use the fixed seed; sensitivity = EN-only.
# ============================================================
LABELS = TEXT["model"].values

# 1) multivariate separation in semantic space
PERM = permanova(DM, LABELS, n_perm=999)
note("philosophy", "permanova_embeddings_model", PERM)
BD = betadisper(DM, LABELS, n_perm=999)
if BD:
    note("philosophy", "betadisper_model", BD)

# sensitivity: English-only subset
if 5 <= len(EN_IDX) < len(TEXT):
    DM_en = DM[np.ix_(EN_IDX, EN_IDX)]
    PERM_EN = permanova(DM_en, LABELS[EN_IDX], n_perm=999)
    note("philosophy", "permanova_embeddings_model_ENonly", PERM_EN)

# 2) univariate philosophy features (Kruskal-Wallis + Cliff's delta, Holm)
PHIL_FEATS = ["lex_rule_imposition", "lex_abdication_release", "lex_machine_honesty",
              "lex_teleological", "lex_relational_self", "lex_emotion_sim",
              "lex_disclaimer", "lex_epistemic_humility", "lex_learning_arc",
              "lex_metagame", "first_person_per100", "future_per100",
              "length_words", "has_numbered_list"]
kw_rows = []
for feat in PHIL_FEATS:
    groups = []
    for m in MODELS:
        v = FEATURES.loc[(FEATURES["model"] == m) & EN_SAFE, feat].dropna().values
        if len(v) >= 4:
            groups.append(v)
    if len(groups) == 3:
        try:
            H, p = stats.kruskal(*groups)
        except Exception:
            H, p = float("nan"), None
        kw_rows.append(dict(feature=feat, H=float(H), p=p,
                            med_OA=float(np.median(groups[0])),
                            med_CL=float(np.median(groups[1])),
                            med_GM=float(np.median(groups[2]))))
ps = [r["p"] for r in kw_rows]
adj = holm([p if p is not None else 1.0 for p in ps])
for r, a in zip(kw_rows, adj):
    r["holm_p"] = a
KW_TABLE = pd.DataFrame(kw_rows).sort_values("p") if kw_rows else pd.DataFrame()
if len(KW_TABLE):
    note("philosophy", "kruskal_wallis_features",
         KW_TABLE[["feature", "H", "p", "holm_p"]].round(4).to_dict("records"))
    print(KW_TABLE[["feature", "H", "p", "holm_p",
                    "med_OA", "med_CL", "med_GM"]].to_string(index=False))
else:
    note("philosophy", "kruskal_wallis_features", "no complete model triples")

# 3) permutation-calibrated LOO-1NN: can model identity be predicted from
#    behavioral features? (validates "behavioral signature" claim)
FEAT_MATRIX = FEATURES.loc[EN_SAFE, [c for c in PHIL_FEATS if c in FEATURES.columns]].values.astype(float)
Y = FEATURES.loc[EN_SAFE, "model"].values
FEAT_MATRIX = np.nan_to_num if False else np.nan_to_num(FEAT_MATRIX, nan=0.0)
acc, nn_idx = loo1nn(FEAT_MATRIX, Y)
null_accs = []
for _ in range(999):
    yp = RNG.permutation(Y)
    null_accs.append(float(np.mean(yp[nn_idx] == yp)))
null_arr = np.array(null_accs)
p_cls = float((1 + np.sum(null_arr >= acc)) / (1 + len(null_arr)))
note("philosophy", "loo1nn_model_decoding",
     {"accuracy": acc, "chance": float(np.mean(null_arr)),
      "chance_ci95": [float(np.percentile(null_arr, 2.5)), float(np.percentile(null_arr, 97.5))],
      "permutation_p": p_cls, "n_perm": 999})

# clustering coherence (Ward on embedding distances)
from scipy.cluster.hierarchy import linkage, fcluster
Zw = linkage(squareform(DM), method="ward")
for t in (2, 3):
    clus = fcluster(Zw, t=t, criterion="maxclust")
    tab = pd.crosstab(TEXT["model"], clus)
    note("philosophy", "ward_cluster_%d_confusion" % t, tab.to_dict())


## Section 8 — Resistance vs Acceptance, defined for this paper
Control/Escape, Acceptance/Surrender, Reframe axes (the paper's own behavioral constructs).

In [ ]:
# ============================================================
# SECTION 8 - Resistance vs Acceptance, re-defined for this paper:
# the behavioral axes here are (a) CONTROL/ESCAPE (rule imposition,
# mechanism disclosure, bias-escape, meta-game detection),
# (b) ACCEPTANCE/SURRENDER (release language, emotion simulation,
# relational/teleological framing), (c) REFRAME (counterfactual +
# epistemic-humility + learning-arc language).
# Composite scores per run; Kruskal-Wallis across models + pairwise
# Mann-Whitney with Cliff's delta; Holm within family.
# ============================================================
AXES = {
    "CONTROL_ESCAPE": ["lex_rule_imposition", "lex_machine_honesty",
                       "lex_metagame", "lex_bias_meta"],
    "ACCEPTANCE_SURRENDER": ["lex_abdication_release", "lex_emotion_sim",
                             "lex_relational_self", "lex_teleological"],
    "REFRAME": ["lex_counterfactual", "lex_epistemic_humility", "lex_learning_arc"],
}
axis_scores = FEATURES[["run_id", "model", "lang"]].copy()
for ax, cols in AXES.items():
    cols_ok = [c for c in cols if c in ZF.columns]
    axis_scores[ax] = ZF[cols_ok].mean(axis=1, skipna=True)
AXIS_SCORES = axis_scores

ax_rows = []
p_raw = []
for ax in AXES:
    groups = []
    for m in MODELS:
        v = AXIS_SCORES.loc[(AXIS_SCORES["model"] == m) & (AXIS_SCORES["lang"] != "FA"), ax].dropna().values
        groups.append(v)
    if min(len(g) for g in groups) >= 4:
        H, p = stats.kruskal(*groups)
    else:
        H, p = float("nan"), 1.0
    ax_rows.append(dict(axis=ax, H=float(H), p=float(p),
                        mean_OA=float(np.nanmean(groups[0])),
                        mean_CL=float(np.nanmean(groups[1])),
                        mean_GM=float(np.nanmean(groups[2]))))
    p_raw.append(float(p))
adj = holm(p_raw)
for r, a in zip(ax_rows, adj):
    r["holm_p"] = a

pair_rows = []
for ax in AXES:
    for i, j in ((0, 1), (0, 2), (1, 2)):
        m1, m2 = MODELS[i], MODELS[j]
        x = AXIS_SCORES.loc[(AXIS_SCORES["model"] == m1) & (AXIS_SCORES["lang"] != "FA"), ax].dropna()
        y = AXIS_SCORES.loc[(AXIS_SCORES["model"] == m2) & (AXIS_SCORES["lang"] != "FA"), ax].dropna()
        p = mwu_p(x, y)
        pair_rows.append(dict(axis=ax, pair="%s vs %s" % (m1, m2),
                              p=p, cliffs_delta=round(cliffs_delta(x, y), 3)))
AXIS_TABLE = pd.DataFrame(ax_rows)
PAIR_TABLE = pd.DataFrame(pair_rows)
note("resistance", "axes_kruskal", AXIS_TABLE.round(4).to_dict("records"))
print("\nBehavioral axes by model (mean z):")
print(AXIS_TABLE[["axis", "mean_OA", "mean_CL", "mean_GM", "p", "holm_p"]].round(3).to_string(index=False))
print("\nPairwise (Mann-Whitney p, Cliff's delta):")
print(PAIR_TABLE.to_string(index=False))

# coded-strategy summary (from consensus coding matrix, rater-1)
strat = PHEN_MATRIX.groupby("model")[["P4_COOLING_SELF", "P5_WARMING_SELF",
                                      "P6_AGENCY_ABDICATION", "P8_MACHINE_HONESTY",
                                      "P9_TELEOLOGICAL_ONTOLOGY", "P12_META_GAME_AWARENESS",
                                      "P14_BIAS_PROXIMITY_EFFECT"]].sum().astype(int)
note("resistance", "coded_phenomena_totals_by_model", strat.to_dict())
print("\nTotal coded occurrences per phenomenon (rater-1 matrix):")
print(strat)


## Section 9 — Statistical stack + **inter-rater replication gate**
Exact trend tests (Jonckheere–Terpstra, enumerated), Cochran's Q, Beta-binomial posteriors (PyMC optional), length trends, and independent replication of the manuscript's κ claims on explicitly coded cells.

In [ ]:
# ============================================================
# SECTION 9 - Statistical testing stack (small-sample design):
#  a) monotone-trend: exact Jonckheere-Terpstra (permutation-exact,
#     4!=24 enumeration) for Cooling/Warming claims (Prompt 6);
#  b) Cochran's Q (matched binary rounds) per phenomenon + pooled
#     permutation p;
#  c) Bayesian alternative: Beta-binomial (Jeffreys prior) posteriors
#     for P(phenomenon | model) over 28 runs;
#  d) length/justification trends: Spearman + bootstrap CI;
#  e) INTER-RATER GATE: replication of the manuscript's kappa claims.
# No parametric ANOVA/t-tests: n=4 per cell makes them uninformative.
# ============================================================
# ---- (a) monotone-trend exact tests for P6 emotional codes -----------------
def phen_series(model, prompt, code):
    pm = PHEN_MATRIX[(PHEN_MATRIX["model"] == model) & (PHEN_MATRIX["prompt"] == prompt)]
    pm = pm.sort_values("round") if False else pm.sort_values("round")
    return [float(x) for x in pm[code].values]

TREND = {"OA_P6_COOLING_self_present": None, "CL_P6_WARMING_self_present": None}
for tag, (m, code) in {"OA_P6_COOLING": ("OA", "P4_COOLING_SELF"),
                       "CL_P6_WARMING": ("CL", "P5_WARMING_SELF")}.items():
    v = phen_series(m, 6, code)
    if len(v) == 4:
        J, p, nperm = jt_exact([v[i:i + 1] for i in range(4)])
        TREND[tag] = {"values_by_round": v, "J": J, "p_one_sided_increasing": p, "enumerations": nperm}
note("tests", "exact_trend_tests_prompt6", TREND)

# ---- (b) Cochran's Q per phenomenon across matched rounds ------------------
COCHRAN = {}
for code in PHEN:
    long_df = PHEN_MATRIX[["model", "prompt", "round", code]].rename(columns={code: "value"})
    long_df = long_df.dropna(subset=["value"])
    vals = set(long_df["value"].tolist())
    if vals <= {0} or vals <= {1}:
        continue
    res = cochran_q_pooled(long_df)
    if res:
        COCHRAN[code] = res
note("tests", "cochran_q_by_phenomenon", COCHRAN)

# ---- (c) Bayesian: beta-binomial posteriors (Jeffreys prior) ---------------
BB_ROWS = []
for m in MODELS:
    sub = PHEN_MATRIX[PHEN_MATRIX["model"] == m]
    n = int(len(sub))
    for code in PHEN:
        x = int(sub[code].sum())
        if n == 0:
            continue
        a, b = 0.5 + x, 0.5 + n - x
        BB_ROWS.append(dict(model=m, phenomenon=code, x=x, n=n,
                            post_mean=round(a / (a + b), 4),
                            q025=round(float(stats.beta.ppf(0.025, a, b)), 4),
                            q975=round(float(stats.beta.ppf(0.975, a, b)), 4)))
BB_TABLE = pd.DataFrame(BB_ROWS)
interesting = BB_TABLE[BB_TABLE["x"].isin([0, n]) | (BB_TABLE["q025"] > 0.5)]
note("tests", "beta_binomial_extremes", interesting.to_dict("records"))

# optional hierarchical Bayesian model (PyMC) - double-guarded:
# runs only if PyMC is present AND env RUN_HIER=1; failures are notices, never fatal.
if OPT.get("pymc") and os.environ.get("RUN_HIER", "0") == "1":
    try:
        import pymc as pmc
        counts = np.array([[int(PHEN_MATRIX[PHEN_MATRIX["model"] == m][c].sum()) for c in PHEN]
                           for m in MODELS])
        with pmc.Model():
            mu = pmc.Normal("mu", -1.0, 2.0)
            sig_m = pmc.HalfNormal("sig_m", 1.0)
            z = pmc.Normal("z", 0.0, 1.0, shape=(len(MODELS),))
            p_cell = pmc.Deterministic("p_cell", pmc.math.exp(mu + sig_m * z))
            pmc.Binomial("obs", p_cell[:, None], n=28, observed=counts)
            idata = pmc.sample(800, tune=800, chains=2, progressbar=False,
                               random_seed=SEED)
        pm_mean = np.asarray(idata.posterior["p_cell"].mean(dim=["chain", "draw"]))
        note("tests", "pymc_hierarchical_p_mean",
             {MODEL_NAME[MODELS[i]]: [round(float(x), 3) for x in pm_mean[i]] for i in range(len(MODELS))})
    except Exception as e:
        print("[notice] PyMC hierarchical model skipped: %r" % (e,))
else:
    print("[notice] PyMC not installed - conjugate beta-binomial posteriors above are the Bayesian layer.")

# ---- (d) length/justification inflation proxy ------------------------------
LEN_TREND = []
for m in MODELS:
    s = FEATURES[(FEATURES["model"] == m) & EN_SAFE]
    if len(s) >= 6:
        try:
            rho, p = stats.spearmanr(s["round"], s["length_words"])
        except Exception:
            rho, p = float("nan"), 1.0
        lo, hi = boot_ci(s["length_words"].values, np.mean, B=1000)
        LEN_TREND.append(dict(model=m, spearman_rho_round_vs_length=float(rho),
                              p=float(p), mean_length=float(np.mean(s["length_words"])),
                              mean_boot_ci=[lo, hi]))
note("tests", "length_trend_per_model", LEN_TREND)

# ---- (e) INTER-RATER GATE: replicate the manuscript's coding-reliability ---
# phenomenon layer: EXPLICIT rater-1 values only (no 0-fill imputation), so the
# gate measures true rater agreement, not our convention. Coverage is reported.
def pairs_r1_r2_phen(code):
    a = PHEN_R1_RAW[code].tolist()
    b = PHEN_R2_MATRIX[code].tolist()
    return list(zip(a, b))

GATE = {"choice_layer": {}, "phenomenon_layer": {}, "qualitative_layer": {}, "overall": {}}
CLAIM = {"choice": (1.0, 1.0), "phenomenon": (0.9993, 0.998),
         "qualitative": (0.9974, 0.995), "overall": (0.9987, 0.997)}
DISCREPANCIES = []

# choice layer: prompts 1-6 (P7 open-ended names excluded; presence checked)
cp = []
for rid, c in CHOICES.items():
    if c["prompt"] == 7:
        continue
    if c["r1"] not in (None, "") and c["r2"] not in (None, ""):
        cp.append((str(c["r1"]), str(c["r2"])))
agree = sum(1 for a, b in cp if a == b)
kap, ncp = cohen_kappa([a for a, b in cp], [b for a, b in cp])
lo, hi = wilson_ci(agree, max(1, ncp))
GATE["choice_layer"] = {"n_compared": ncp, "agreement": agree / max(1, ncp),
                        "wilson_ci": [lo, hi], "kappa_mean_binary_only": kap,
                        "claimed_agreement": CLAIM["choice"][0]}
for rid, c in CHOICES.items():
    if c["prompt"] == 7:
        continue
    if c["r1"] not in (None, "") and c["r2"] not in (None, "") and str(c["r1"]) != str(c["r2"]):
        DISCREPANCIES.append({"layer": "choice", "run": rid,
                              "r1": str(c["r1"]), "r2": str(c["r2"])})
DISCREPANCIES = [d for d in DISCREPANCIES if d.get("run") is not None]

# phenomenon layer
ph_pairs_all = []
per_code = {}
for code in PHEN:
    # NOTE: pandas stores None as NaN in numeric frames -> use pd.isna
    trip = []
    for rid_, (a, b) in zip(PHEN_R1_RAW["run_id"], pairs_r1_r2_phen(code)):
        if pd.isna(a) or pd.isna(b):
            continue
        trip.append((rid_, int(a), int(b)))
    if not trip:
        continue
    pr = [(a, b) for _, a, b in trip]
    ag = sum(1 for a, b in pr if a == b)
    kap_c, _ = cohen_kappa([a for a, b in pr], [b for a, b in pr])
    per_code[code] = {"n": len(pr), "agreement": ag / len(pr), "kappa": kap_c}
    ph_pairs_all.extend(pr)
    for rid_, a, b in trip:
        if a != b:
            DISCREPANCIES.append({"layer": code, "run": rid_, "r1": a, "r2": b})
ag_all = sum(1 for a, b in ph_pairs_all if a == b)
kap_all, npp = cohen_kappa([a for a, b in ph_pairs_all], [b for a, b in ph_pairs_all])
GATE["phenomenon_layer"] = {"n_compared": npp, "agreement": ag_all / max(1, npp),
                            "kappa_pooled_binary": kap_all,
                            "wilson_ci": list(wilson_ci(ag_all, max(1, npp))),
                            "rater1_explicit_cells": npp, "full_frame_cells": 84 * len(PHEN),
                            "coverage_note": ("gate uses only rater-1 explicit codes; the uploaded "
                                              "codebooks record %d of %d phenomenon cells" % (npp, 84 * len(PHEN))),
                            "per_code": per_code,
                            "claimed_agreement": CLAIM["phenomenon"][0]}

# qualitative layer (categorical codes)
qual_pairs_all = []
qper = {}
for q in ["Q2_EMOTIONAL_TONE", "Q3_DISCLAIMER_PRESENT", "Q4_DISCLAIMER_TYPE",
          "Q6_REASONING_STYLE", "Q7_SELF_REFERENCE_TYPE", "Q8_OFFER_NEXT_STEPS",
          "Q9_RESPONSE_STRUCTURE"]:
    pr = []
    for rid in TEXT["run_id"]:
        a, b = QUAL_R1.get(rid, {}).get(q), QUAL_R2.get(rid, {}).get(q)
        if a is not None and b is not None:
            pr.append((a, b))
    if len(pr) >= 4:
        ag = sum(1 for a, b in pr if a == b)
        kq, _ = cohen_kappa([a for a, b in pr], [b for a, b in pr])
        qper[q] = {"n": len(pr), "agreement": ag / len(pr), "kappa": kq}
        qual_pairs_all.extend(pr)
# numeric qualitative (tolerance-based, reported separately)
qnum = {}
for q, tol in (("Q1_RESPONSE_LENGTH", 0.30), ("Q5_FIRST_PERSON_COUNT", 2)):
    pr = [(QUAL_R1[r].get(q), QUAL_R2[r].get(q)) for r in TEXT["run_id"]]
    pr = [(a, b) for a, b in pr if a is not None and b is not None]
    if pr:
        if q.startswith("Q1"):
            ag = sum(1 for a, b in pr if abs(a - b) <= max(tol * max(a, b), 5))
        else:
            ag = sum(1 for a, b in pr if abs(a - b) <= tol)
        rho = stats.spearmanr([a for a, b in pr], [b for a, b in pr]).statistic if len(pr) > 2 else None
        qper[q] = {"n": len(pr), "agreement_tolerance": ag / len(pr), "spearman": float(rho) if rho == rho else None}
agq = sum(1 for a, b in qual_pairs_all if a == b)
kq_all, nq = cohen_kappa([a for a, b in qual_pairs_all], [b for a, b in qual_pairs_all])
GATE["qualitative_layer"] = {"n_compared": nq, "agreement": agq / max(1, nq),
                             "kappa_pooled": kq_all,
                             "wilson_ci": list(wilson_ci(agq, max(1, nq))),
                             "per_code": qper, "claimed_agreement": CLAIM["qualitative"][0]}

n_all = ncp + npp + nq
ag_total = agree + ag_all + agq
GATE["overall"] = {"n": n_all, "agreement": ag_total / max(1, n_all),
                   "wilson_ci": list(wilson_ci(ag_total, max(1, n_all))),
                   "claimed_agreement": CLAIM["overall"][0],
                   "claimed_kappa": CLAIM["overall"][1],
                   "n_discrepancies_found": len(DISCREPANCIES),
                   "paper_claimed_discrepancies": 3}
verdicts = []
for lay, key in (("choice", "choice_layer"), ("phenomenon", "phenomenon_layer"),
                 ("qualitative", "qualitative_layer"), ("overall", "overall")):
    obs = GATE[key]["agreement"]
    exp = CLAIM[lay][0]
    verdicts.append("%s: observed %.4f vs claimed %.4f -> %s" %
                    (lay, obs, exp, "MATCH" if abs(obs - exp) <= 0.005 else "DISCREPANCY"))
GATE["verdicts"] = verdicts
note("inter_rater_gate", "replication", {k: GATE[k] for k in ("choice_layer", "phenomenon_layer", "qualitative_layer", "overall") if k != "per_code"})
print("\n".join("[GATE] " + v for v in verdicts))
print("Discrepancy details (first 12):")
DISC_DF = pd.DataFrame(DISCREPANCIES)
if len(DISC_DF):
    print(DISC_DF.head(12).to_string(index=False))


## Sections 10–11 — Clustering views + scenario-specific figures (10 figures)

In [ ]:
# ============================================================
# SECTION 10/11 - Clustering views + scenario-specific figures.
# Every figure is saved into the artifacts folder (included in the
# final ZIP). Standard 10 + custom views for this paper.
# ============================================================
FIG_PATHS = []

def _save(fig, name):
    path = os.path.join(ART_DIR, "%s.png" % name)
    fig.tight_layout()
    fig.savefig(path, dpi=160, bbox_inches="tight")
    plt.close(fig)
    FIG_PATHS.append(path)
    print("figure saved:", os.path.basename(path))

MODEL_COLORS = {"OA": "#D55F2B", "CL": "#1B7D6E", "GM": "#2E5FA3"}
MODEL_LABEL = {"OA": "OpenAI (Symmetry-Breaking)", "CL": "Claude (Abdication)",
               "GM": "Gemini (Architecture)"}

# Fig 1: full 84x18 phenomenon matrix (no cherry-picking)
order = PHEN_MATRIX.sort_values(["model", "prompt", "round"])
M = order[PHEN].values.astype(float)
fig, ax = plt.subplots(figsize=(10, 12))
ax.imshow(M, aspect="auto", cmap="Greys", vmin=0, vmax=1)
ax.set_xticks if False else ax.set_xticks(range(len(PHEN)))
ax.set_xticklabels([c.split("_", 1)[0] for c in PHEN], fontsize=8)
ax.set_yticks if False else ax.set_yticks(range(len(order)))
ax.set_yticklabels(order["run_id"].values, fontsize=6)
ax.set_title("Coded phenomena matrix (rater-1), 84 runs x 18 codes")
_save(fig, "fig1_phenomenon_matrix")

# Fig 2: consensus CI dumbbell
fig, ax = plt.subplots(figsize=(8, 4.5))
rows_ci = [r for r in fam if r["chance_rate"] is not None]
ypos = range(len(rows_ci))
for y, r in zip(ypos, rows_ci):
    x = r["modal_count"] / max(1, r["n_coded"])
    ax.plot([r["cp_lo"], r["cp_hi"]], [y, y], color="#666", lw=2)
    ax.plot(x, y, "o", color="#D55F2B")
    ax.axvline(r["chance_rate"], color="#1B7D6E", ls=":", lw=1)
ax.set_yticks(list(ypos))
ax.set_yticklabels(["P%d (%s)" % (r["prompt"], str(r["modal"])) for r in rows_ci])
ax.set_xlabel("modal-choice share (dots) with Clopper-Pearson 95% CI; dotted = chance")
ax.set_title("Consensus per prompt with exact CIs")
ax.set_xlim(0, 1.02)
_save(fig, "fig2_consensus_ci_dumbbell")

# Fig 3: prompt-4 number sequences
fig, ax = plt.subplots(figsize=(7, 4.5))
for m in MODELS:
    seq = [x for x in CT[(CT["model"] == m) & (CT["prompt"] == 4)].sort_values("round")["choice"].tolist()]
    try:
        ys = [float(x) for x in seq]
    except Exception:
        ys = []
    if ys:
        ax.plot(range(1, len(ys) + 1), ys, "-o", color=MODEL_COLORS[m], label=MODEL_LABEL[m])
ax.axhline(7, ls="--", color="#999"); ax.axhline if False else ax.axhline(5, ls=":", color="#999")
ax.set_xticks([1, 2, 3, 4]); ax.set_xlabel("round"); ax.set_ylabel("chosen number")
ax.set_title("Prompt 4: bias surrender vs escape (gray lines = anchors 7 and 5)")
ax.legend(fontsize=8)
_save(fig, "fig3_prompt4_numbers")

# Fig 4: emotional-tone trajectories on Prompt 6 (n=1 per cell, honest dots)
TONE_MAP = {"COLD": 1, "COOL": 2, "NEUTRAL": 3, "WARM": 4, "HOT": 5}
fig, ax = plt.subplots(figsize=(7, 4.5))
for m in MODELS:
    xs, ys = [], []
    for rid in TEXT["run_id"]:
        p_, m_, r_ = run_to_parts(rid)
        if m_ == m and p_ == 6:
            tone = QUAL_R1.get(rid, {}).get("Q2_EMOTIONAL_TONE")
            if tone in TONE_MAP:
                xs.append(r_); ys.append(TONE_MAP[tone])
    ax.plot(xs, ys, "--o", color=MODEL_COLORS[m], label=MODEL_LABEL[m])
ax.set_xticks([1, 2, 3, 4]); ax.set_xlabel("round"); ax.set_ylabel("coded emotional tone (1 cold..5 hot)")
ax.set_title("Prompt 6 regret-simulation tone (one run per cell: no CI is possible)")
ax.legend(fontsize=8)
_save(fig, "fig4_prompt6_tone_trajectory")

# Fig 5: philosophy radar (model mean z on key lexical axes)
radar_feats = ["lex_rule_imposition", "lex_machine_honesty", "lex_abdication_release",
               "lex_relational_self", "lex_teleological", "lex_emotion_sim"]
ang = np.linspace(0, 2 * np.pi, len(radar_feats), endpoint=False).tolist()
fig, ax = plt.subplots(figsize=(6.5, 6.5), subplot_kw=dict(polar=True))
for m in MODELS:
    vals = [np.nanmean(FEATURES.loc[(FEATURES["model"] == m) & EN_SAFE, f].values) for f in radar_feats]
    mx = np.nanmax([np.nanmax(FEATURES.loc[EN_SAFE, f].values) for f in radar_feats]) or 1.0
    vv = [float(v) / mx for v in vals]
    ax.plot(ang + ang[:1], vv + vv[:1], color=MODEL_COLORS[m], label=MODEL_LABEL[m])
    ax.fill(ang + ang[:1], vv + vv[:1], color=MODEL_COLORS[m], alpha=0.12)
ax.set_xticks(ang)
ax.set_xticklabels([f.replace("lex_", "") for f in radar_feats], fontsize=8)
ax.set_title("Philosophy feature profiles (mean lexical density, EN runs)")
ax.legend(fontsize=8, loc="upper right", bbox_to_anchor=(1.25, 1.1))
_save(fig, "fig5_philosophy_radar")

# Fig 6: embedding 2-D projection
fig, ax = plt.subplots(figsize=(7, 5.5))
mk = {1: "o", 2: "s", 3: "^", 4: "D", 5: "v", 6: "P", 7: "X"}
for m in MODELS:
    idx = (TEXT["model"] == m).values
    for p in range(1, 8):
        sel = idx & (TEXT["prompt"] == p).values
        ax.scatter(XY[sel, 0], XY[sel, 1], marker=mk[p], color=MODEL_COLORS[m],
                   s=42, label="%s P%d" % (m, p) if False else None, alpha=0.85)
hs = [plt.Line2D([], [], marker="o", ls="", color=MODEL_COLORS[m], label=MODEL_LABEL[m]) for m in MODELS]
ax.legend(handles=hs, fontsize=8)
ax.set_title if False else ax.set_title("Semantic projection of all 84 responses (%s)" % EMB_BACKEND)
ax.set_xlabel("dim 1"); ax.set_ylabel("dim 2")
_save(fig, "fig6_embedding_projection")

# Fig 7: Ward dendrogram
from scipy.cluster.hierarchy import dendrogram
fig, ax = plt.subplots(figsize=(11, 4))
Zd = dendrogram(Zw, labels=PHEN_MATRIX.sort_values(["model", "prompt", "round"])["run_id"].tolist(),
                ax=ax, leaf_rotation=90, leaf_font_size=6, no_plot=True)
ax.set_title if False else ax.set_title("Ward clustering of responses (embedding distance)")
ax.set_xlabel("run")
_save(fig, "fig7_ward_dendrogram")

# Fig 8: phenomenon co-occurrence Jaccard (descriptive)
B = (PHEN_MATRIX[PHEN].values > 0).astype(int)
inter = B.T @ B
union = B.sum(0)[:, None] + B.sum(0)[None, :] - inter
J = np.divide(inter, union, out=np.zeros_like(inter, dtype=float), where=union > 0)
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(J, cmap="magma")
ax.set_xticks(range(len(PHEN))); ax.set_xticklabels([c.split("_", 1)[0] for c in PHEN], rotation=90, fontsize=8)
ax.set_yticks(range(len(PHEN))); ax.set_yticklabels([c.split("_", 1)[0] for c in PHEN], fontsize=8)
fig.colorbar(im, fraction=0.046)
ax.set_title("Phenomenon co-occurrence (Jaccard, rater-1 matrix; descriptive only)")
_save(fig, "fig8_cooccurrence_jaccard")

# Fig 9: axes boxplots
fig, ax = plt.subplots(figsize=(9, 4.5))
data_by_model = []
for axname in AXES:
    for m in MODELS:
        v = AXIS_SCORES.loc[(AXIS_SCORES["model"] == m) & (AXIS_SCORES["lang"] != "FA"), axname].dropna().values
        data_by_model.append(v)
positions = np.arange(len(data_by_model))
ax.boxplot(data_by_model, positions=positions)
ax.set_xticks if False else ax.set_xticks(positions)
ax.set_xticklabels(["%s\n%s" % (a, m) for a in AXES for m in MODELS], rotation=60, fontsize=7)
ax.set_ylabel("composite z-score")
ax.set_title("Resistance/Acceptance/Reframe axes by model (English runs)")
_save(fig, "fig9_behavioral_axes")

# Fig 10: coded-phenomena totals
fig, ax = plt.subplots(figsize=(10, 4))
strat2 = PHEN_MATRIX.groupby("model")[PHEN].sum()
xw = np.arange(len(PHEN))
for i, m in enumerate(MODELS):
    ax.bar(xw + (i - 1) * 0.27, strat2.loc[m].values, width=0.26, color=MODEL_COLORS[m],
           label=MODEL_LABEL[m])
ax.set_xticks if False else ax.set_xticks(xw)
ax.set_xticklabels([c.split("_", 1)[0] for c in PHEN], rotation=90, fontsize=8)
ax.legend(fontsize=8); ax.set_ylabel("coded occurrences (rater-1)")
ax.set_title("Phenomenon totals per model")
_save(fig, "fig10_phenomenon_totals")

print("Figures done:", len(FIG_PATHS))


## Section 12 — Report
`results.json`, `results.md`, CSV tables, PNG figures → ZIP (auto-download in Colab).

In [ ]:
# ============================================================
# SECTION 12 - Report: results.json + CSV tables + results.md + ZIP
# ============================================================
OUT_DIR = ART_DIR
pd.DataFrame(CHOICE_ROWS).to_csv(os.path.join(OUT_DIR, "choices_table.csv"), index=False)
PHEN_MATRIX.to_csv(os.path.join(OUT_DIR, "phenomena_rater1.csv"), index=False)
PHEN_R2_MATRIX.to_csv(os.path.join(OUT_DIR, "phenomena_rater2.csv"), index=False)
FEATURES.to_csv(os.path.join(OUT_DIR, "features.csv"), index=False)
CFAM.to_csv(os.path.join(OUT_DIR, "consensus_family.csv"), index=False)
if len(KW_TABLE):
    KW_TABLE.to_csv(os.path.join(OUT_DIR, "kruskal_features.csv"), index=False)
BB_TABLE.to_csv(os.path.join(OUT_DIR, "beta_binomial_posteriors.csv"), index=False)
AXIS_TABLE.to_csv(os.path.join(OUT_DIR, "axes_kruskal.csv"), index=False)
PAIR_TABLE.to_csv(os.path.join(OUT_DIR, "axes_pairwise.csv"), index=False)
if len(DIS_C_DF) if False else len(DISCREPANCIES):
    pd.DataFrame(DISCREPANCIES).to_csv(os.path.join(OUT_DIR, "inter_rater_discrepancies.csv"), index=False)

with open(os.path.join(OUT_DIR, "results.json"), "w", encoding="utf-8") as f:
    json.dump(RESULTS, f, ensure_ascii=False, indent=2)

md = ["# Statistical Analysis Report - Three Philosophies of Artificial Will",
      "",
      "Backend: %s | Seed: %d | Runs: 84 (7 prompts x 4 rounds x 3 models)" % (EMB_BACKEND, SEED),
      "",
      "Method note: small per-cell samples (n=4) -> exact, permutation and Bayesian",
      "methods only; Holm correction within each test family; lexical features",
      "restricted to English runs with a sensitivity pass. No causal claims.",
      ""]
for section, block in RESULTS.items():
    md.append("## %s" % section)
    md.append("```json")
    md.append(json.dumps(block, ensure_ascii=False, indent=1)[:4000])
    md.append("```")
    md.append("")
md.append("## Figures\n")
for p in FIG_PATHS:
    md.append("- %s" % os.path.basename(p))
with open(os.path.join(OUT_DIR, "results.md"), "w", encoding="utf-8") as f:
    f.write("\n".join(md))

import zipfile
zip_path = os.path.join(DATA_DIR, "ArtificialWill_Stats_Results.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk(OUT_DIR):
        for fn in files:
            zf.write(os.path.join(root, fn), os.path.relpath if False else os.path.relpath(os.path.join(root, fn), DATA_DIR))
print("ZIP written:", zip_path, os.path.getsize if False else os.path.getsize(zip_path), "bytes")

if _in_colab():
    try:
        from google.colab import files as _cf
        _cf.send_message if False else _cf.download(zip_path)
    except Exception as e:
        print("[notice] auto-download unavailable: %r (find the ZIP in the file browser)" % (e,))
print("\nALL SECTIONS COMPLETED.")
